# Attention Head Clustering

Group attention heads by behavioral similarity: pattern similarity,
output direction clustering, functional fingerprints, and redundancy.

## Why This Matters

Attention head clustering reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_clustering import (
    head_pattern_similarity, head_output_direction_clustering,
    head_functional_fingerprint, head_redundancy_score,
    attention_head_clustering_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Pattern Similarity

How similar are attention heads based on their attention patterns?

In [ ]:
for layer in range(4):
    result = head_pattern_similarity(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_sim={result['mean_similarity']:.4f}")
    for p in result['per_pair']:
        print(f"    H{p['head_a']}-H{p['head_b']}: {p['similarity']:.4f}")

## Output Direction Clustering

Heads with similar output directions write to the same subspace.

In [ ]:
for layer in range(4):
    result = head_output_direction_clustering(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_out_sim={result['mean_similarity']:.4f}")

## Functional Fingerprints

Characterize each head by entropy, peak attention, diagonal score.

In [ ]:
result = head_functional_fingerprint(model, tokens, layer=0)
for p in result['per_head']:
    print(f"  H{p['head']}: entropy={p['entropy']:.4f}, "
          f"max_wt={p['max_weight']:.4f}, diag={p['diagonal_score']:.4f}")

## Redundancy Scores

In [ ]:
for layer in range(4):
    result = head_redundancy_score(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_redundant']} redundant heads")
    for p in result['per_head']:
        flag = ' [RED]' if p['is_redundant'] else ''
        print(f"    H{p['head']}: max_sim={p['max_similarity']:.4f} "
              f"(most similar to H{p['most_similar_to']}){flag}")

## Summary

In [ ]:
result = attention_head_clustering_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean_sim={p['mean_pattern_similarity']:.4f}, "
          f"redundant={p['n_redundant']}")